# Evaluation on Large Images with Patchification

This notebook demonstrates how to:
1. Load trained models
2. Patch large images into smaller tiles
3. Run predictions on patches
4. Reconstruct full-size predictions
5. Visualize and evaluate results

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import json

from utils.patchify import (
    predict_large_image,
    patchify_image,
    unpatchify_image
)
from models.model_factory import ModelFactory
from cilia_utils.utils import normalize_channel

## 1. Configuration

In [ ]:
# Model configuration
MODEL_CHECKPOINT = '../outputs/c1_unet_tiny/best_model.pth'  # Update this path
ARCHITECTURE = 'unet_tiny'
CHANNEL_MODE = 'c1_only'  # or 'c2_only' or 'dual'
C1_IDX = 0
C2_IDX = 1
DUAL_MODE = 'average'

# Inference configuration
PATCH_SIZE = 512  # Must match training image size
OVERLAP = 64      # Overlap between patches
BATCH_SIZE = 8
THRESHOLD = 0.5
BLEND_MODE = 'average'  # 'average', 'max', or 'first'

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 2. Load Model

In [ ]:
def load_model(checkpoint_path, architecture, device='cuda'):
    """Load trained model from checkpoint."""
    model = ModelFactory.create_model(
        architecture,
        num_classes=1,
        pretrained=False
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Loaded model from: {checkpoint_path}")
    print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"Metrics: {checkpoint.get('metrics', 'N/A')}")
    
    return model

model = load_model(MODEL_CHECKPOINT, ARCHITECTURE, DEVICE)

## 3. Load Test Image

In [ ]:
# Load your large test image
# Example: loading from .npy file
test_images_path = '../data/test_images.npy'  # Update this
test_masks_path = '../data/test_masks.npy'    # Update this (optional)

# Load data
test_images = np.load(test_images_path)  # Shape: (N, C, H, W)
print(f"Test images shape: {test_images.shape}")

# Load ground truth if available
try:
    test_masks = np.load(test_masks_path)  # Shape: (N, C, H, W) or (N, H, W)
    print(f"Test masks shape: {test_masks.shape}")
    has_gt = True
except:
    print("No ground truth masks available")
    has_gt = False

## 4. Prepare Image for Inference

In [ ]:
def prepare_image_for_inference(image, channel_mode='c1_only', c1_idx=0, c2_idx=1, dual_mode='average'):
    """
    Prepare multichannel image for model inference.
    
    Args:
        image: (C, H, W) multichannel image
        channel_mode: 'c1_only', 'c2_only', or 'dual'
        c1_idx: Channel 0 index
        c2_idx: Channel 1 index
        dual_mode: 'average', 'zeros', or 'overlay'
    
    Returns:
        RGB image (3, H, W) ready for model
    """
    if channel_mode == 'c1_only':
        c1 = image[c1_idx]
        c1_norm = normalize_channel(c1)
        rgb = np.stack([c1_norm] * 3, axis=0)
    
    elif channel_mode == 'c2_only':
        c2 = image[c2_idx]
        c2_norm = normalize_channel(c2)
        rgb = np.stack([c2_norm] * 3, axis=0)
    
    elif channel_mode == 'dual':
        c1 = normalize_channel(image[c1_idx])
        c2 = normalize_channel(image[c2_idx])
        
        if dual_mode == 'average':
            c3 = (c1 + c2) / 2.0
        elif dual_mode == 'zeros':
            c3 = np.zeros_like(c1)
        elif dual_mode == 'overlay':
            c3 = np.maximum(c1, c2)
        
        rgb = np.stack([c1, c2, c3], axis=0)
    
    return (rgb * 255).astype(np.uint8).astype(np.float32) / 255.0

# Test on first image
test_idx = 0
raw_image = test_images[test_idx]  # (C, H, W)
prepared_image = prepare_image_for_inference(
    raw_image,
    CHANNEL_MODE,
    C1_IDX,
    C2_IDX,
    DUAL_MODE
)

print(f"Raw image shape: {raw_image.shape}")
print(f"Prepared image shape: {prepared_image.shape}")

## 5. Run Prediction with Patchification

In [ ]:
# Predict on the prepared image
pred_mask, pred_binary = predict_large_image(
    model=model,
    image=prepared_image,
    patch_size=PATCH_SIZE,
    overlap=OVERLAP,
    batch_size=BATCH_SIZE,
    blend_mode=BLEND_MODE,
    device=DEVICE,
    threshold=THRESHOLD
)

print(f"Prediction shape: {pred_mask.shape}")
print(f"Binary prediction shape: {pred_binary.shape}")

## 6. Visualize Results

In [ ]:
def visualize_prediction(image, prediction, gt_mask=None, figsize=(20, 10)):
    """
    Visualize prediction results.
    
    Args:
        image: (3, H, W) RGB image
        prediction: (H, W) prediction mask
        gt_mask: (H, W) ground truth mask (optional)
        figsize: Figure size
    """
    n_plots = 3 if gt_mask is not None else 2
    fig, axes = plt.subplots(1, n_plots, figsize=figsize)
    
    # Convert image to HWC for display
    img_display = image.transpose(1, 2, 0)
    img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)
    
    # Input image
    axes[0].imshow(img_display)
    axes[0].set_title('Input Image', fontsize=14)
    axes[0].axis('off')
    
    # Prediction
    axes[1].imshow(prediction, cmap='gray')
    axes[1].set_title('Prediction', fontsize=14)
    axes[1].axis('off')
    
    # Ground truth (if available)
    if gt_mask is not None:
        axes[2].imshow(gt_mask, cmap='gray')
        axes[2].set_title('Ground Truth', fontsize=14)
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize
if has_gt:
    gt_mask = test_masks[test_idx]
    if gt_mask.ndim == 3:
        # Handle multi-channel masks
        if CHANNEL_MODE == 'c1_only':
            gt_mask = gt_mask[C1_IDX]
        elif CHANNEL_MODE == 'c2_only':
            gt_mask = gt_mask[C2_IDX]
        else:  # dual
            gt_mask = np.maximum(gt_mask[C1_IDX], gt_mask[C2_IDX])
    visualize_prediction(prepared_image, pred_binary, gt_mask)
else:
    visualize_prediction(prepared_image, pred_binary)

## 7. Compute Metrics (if ground truth available)

In [ ]:
if has_gt:
    def compute_metrics(pred, gt):
        """Compute segmentation metrics."""
        pred = pred.astype(np.float32)
        gt = gt.astype(np.float32)
        
        # IoU
        intersection = (pred * gt).sum()
        union = pred.sum() + gt.sum() - intersection
        iou = intersection / (union + 1e-8)
        
        # Dice
        dice = (2 * intersection) / (pred.sum() + gt.sum() + 1e-8)
        
        # Precision and Recall
        tp = (pred * gt).sum()
        fp = (pred * (1 - gt)).sum()
        fn = ((1 - pred) * gt).sum()
        
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        
        return {
            'iou': float(iou),
            'dice': float(dice),
            'precision': float(precision),
            'recall': float(recall)
        }
    
    metrics = compute_metrics(pred_binary, gt_mask)
    
    print("\nMetrics:")
    print("="*40)
    for key, value in metrics.items():
        print(f"{key.capitalize():<12}: {value:.4f}")
    print("="*40)

## 8. Batch Processing Multiple Images

In [ ]:
from tqdm import tqdm

def process_batch(images, model, channel_mode, c1_idx, c2_idx, dual_mode, **kwargs):
    """
    Process a batch of images.
    
    Args:
        images: (N, C, H, W) images
        model: Trained model
        channel_mode: Channel mode
        c1_idx, c2_idx: Channel indices
        dual_mode: Dual mode
        **kwargs: Additional arguments for predict_large_image
    
    Returns:
        List of (pred_mask, pred_binary) tuples
    """
    results = []
    
    for i in tqdm(range(len(images)), desc='Processing images'):
        # Prepare image
        prepared = prepare_image_for_inference(
            images[i], channel_mode, c1_idx, c2_idx, dual_mode
        )
        
        # Predict
        pred_mask, pred_binary = predict_large_image(
            model=model,
            image=prepared,
            **kwargs
        )
        
        results.append((pred_mask, pred_binary))
    
    return results

# Example: process first 5 images
# results = process_batch(
#     test_images[:5],
#     model,
#     CHANNEL_MODE,
#     C1_IDX,
#     C2_IDX,
#     DUAL_MODE,
#     patch_size=PATCH_SIZE,
#     overlap=OVERLAP,
#     batch_size=BATCH_SIZE,
#     blend_mode=BLEND_MODE,
#     device=DEVICE,
#     threshold=THRESHOLD
# )

## 9. Save Results

In [ ]:
# Save prediction
output_dir = Path('../outputs/predictions')
output_dir.mkdir(parents=True, exist_ok=True)

# Save as numpy
np.save(output_dir / f'pred_mask_{test_idx}.npy', pred_mask)
np.save(output_dir / f'pred_binary_{test_idx}.npy', pred_binary)

# Save as image (optional)
try:
    from PIL import Image
    pred_img = (pred_binary * 255).astype(np.uint8)
    Image.fromarray(pred_img).save(output_dir / f'pred_{test_idx}.png')
    print(f"Saved prediction to {output_dir}")
except:
    print("PIL not available, saved as .npy only")

## 10. Visualize Multiple Predictions

In [ ]:
def visualize_grid(images, predictions, n_samples=4, figsize=(20, 5)):
    """
    Visualize predictions in a grid.
    """
    n_samples = min(n_samples, len(images))
    fig, axes = plt.subplots(2, n_samples, figsize=figsize)
    
    if n_samples == 1:
        axes = axes[:, None]
    
    for i in range(n_samples):
        # Input
        img = images[i].transpose(1, 2, 0)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        axes[0, i].imshow(img)
        axes[0, i].set_title(f'Image {i}')
        axes[0, i].axis('off')
        
        # Prediction
        axes[1, i].imshow(predictions[i], cmap='gray')
        axes[1, i].set_title(f'Prediction {i}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Example: visualize first 4 predictions if you processed a batch
# prepared_images = [prepare_image_for_inference(img, CHANNEL_MODE, C1_IDX, C2_IDX, DUAL_MODE) 
#                    for img in test_images[:4]]
# predictions = [result[1] for result in results[:4]]
# visualize_grid(prepared_images, predictions)